# Parcial Final - Sistema de Seguridad Inteligente Multi-Cámara (SentinelCV)
## Visión por Computadora con TensorFlow, Keras y PyTorch (CUDA)

Este proyecto desarrolla un sistema de visión artificial para seguridad perimetral capaz de clasificar y detectar automáticamente:
- **Ataque / Agresión física** (`Ataque_Agresion`)
- **Amenaza con Arma de Fuego** (`Arma_De_Fuego`)
- **Amenaza con Arma Blanca** (`Arma_Blanca`)
- **Posible Hurto** (`Hurto_Sospechoso`)
- **Señales de Auxilio a Cámara** (`Senal_Auxilio_Hombre`, `Senal_Auxilio_Mujer`)
- **Escena Normal** (`Escena_Normal`)

Se aplica **Arquitectura Limpia** para desacoplar el modelo entrenado del servicio de inferencia en tiempo real y la consola de monitoreo en Astro.

In [9]:
import os
from pathlib import Path

# Configuración de Keras 3 con backend PyTorch CUDA nativo en Windows
os.environ["KERAS_BACKEND"] = "torch"
os.environ["KERAS_HOME"] = str(Path.cwd() / ".keras")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

print(f"Backend activo de Keras: {keras.backend.backend()}")
print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})")


Backend activo de Keras: torch
PyTorch versión: 2.14.1+cu130
CUDA disponible: True
Dispositivo GPU: NVIDIA GeForce RTX 5050 Laptop GPU (CUDA 13.0)


## 1. Carga del Dataset Real
Se cargan las imágenes reales preparadas desde datasets públicos de Kaggle (`snehilsanyal/weapon-detection-test` y `meetnagadia/human-action-recognition-har-dataset`).

In [10]:
DATASET_DIR = Path.cwd() / "datasets" / "sentinel_real_dataset"
IMG_SIZE = (224, 224)
Input_Shape = IMG_SIZE + (3,)

dataset = keras.utils.image_dataset_from_directory(
    str(DATASET_DIR),
    image_size=IMG_SIZE,
    batch_size=None,
    label_mode='int',
    shuffle=False
)

class_names = dataset.class_names
Num_Clases = len(class_names)
print(f"Clases de escena del dataset ({Num_Clases}):", class_names)

imagenes, etiquetas = [], []
for img, lbl in dataset:
    imagenes.append(img.numpy().astype('uint8'))
    etiquetas.append(int(lbl.numpy()))

X = np.stack(imagenes)
y = np.array(etiquetas)
del imagenes, etiquetas

print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}")


Found 1548 files belonging to 5 classes.
Clases de escena del dataset (5): ['Arma_Blanca', 'Arma_De_Fuego', 'Ataque_Agresion', 'Escena_Normal', 'Hurto_Sospechoso']
X shape: (1548, 224, 224, 3), dtype: uint8
y shape: (1548,)


## 2. Partición Estratificada y Normalización
Se divide en **70% Entrenamiento, 15% Validación y 15% Prueba** con estratificación y normalización `[-1, 1]`.

In [11]:
X_tr, X_temp, y_tr, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val_raw, X_te_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

X_train = (X_tr.astype('float32') / 127.5) - 1.0
X_val = (X_val_raw.astype('float32') / 127.5) - 1.0
X_test = (X_te_raw.astype('float32') / 127.5) - 1.0

print(f"Train: {X_train.shape[0]} muestras")
print(f"Val:   {X_val.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")

conteo_train = np.bincount(y_tr, minlength=Num_Clases)
class_weight = {i: len(y_tr) / (Num_Clases * conteo_train[i]) for i in range(Num_Clases)}
for i, w in class_weight.items():
    print(f"  {class_names[i]:18s} peso = {w:.2f}")


Train: 1083 muestras
Val:   232 muestras
Test:  233 muestras
  Arma_Blanca        peso = 2.41
  Arma_De_Fuego      peso = 0.91
  Ataque_Agresion    peso = 0.86
  Escena_Normal      peso = 0.86
  Hurto_Sospechoso   peso = 0.86


## 3. Data Augmentation
Capas de aumento de datos para robustez ante variaciones de ángulo y brillo.

In [12]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.12, value_range=(-1, 1)),
], name="data_augmentation")


## 4. Modelo 1: CNN Desde Cero (`custom_model`)

In [13]:
custom_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.35),
    layers.Dense(Num_Clases, activation="softmax")
], name="Custom_CNN_SentinelCV")

custom_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
custom_model.summary()


Model: "Custom_CNN_SentinelCV"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,301 (434.77 KB)

 Trainable params: 110,853 (433.02 KB)

 Non-trainable params: 448 (1.75 KB)

## 5. Modelo 2: Transfer Learning con MobileNetV2 (`transfer_model`)

In [14]:
base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

transfer_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(Num_Clases, activation="softmax")
], name="Transfer_Learning_MobileNetV2")

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0008),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
transfer_model.summary()


Model: "Transfer_Learning_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 7, 7, 1280)     │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,435,653 (9.29 MB)

 Trainable params: 175,109 (684.02 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

## 6. Evaluación Comparativa en Holdout Test

In [15]:
import json
with open(Path.cwd() / "models" / "model_metadata.json", "r") as f:
    meta = json.load(f)

print("Comparativa de modelos en Test Set Real:")
df_comp = pd.DataFrame(meta["scene_comparison"]).set_index("Modelo")[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]].round(3)
display(df_comp)


Comparativa de modelos en Test Set Real:


,Accuracy,Precision (macro),Recall (macro),F1 (macro)
Modelo,,,,
CNN desde cero,0.558,0.578,0.581,0.537
Transfer Learning (MobileNetV2),0.459,0.457,0.427,0.426


## 7. Despliegue en Producción
El modelo campeón compilado se encuentra exportado en:
- `models/sentinel_security_model.keras`
- `models/sentinel_gender_model.keras`
- `models/model_metadata.json`

Consumido en tiempo real por el servicio de inferencia `backend_ia` en `http://localhost:8000/api/v1/infer` y la interfaz web en Astro en `http://localhost:4321`.

In [16]:
print("Endpoints de Producción Activos:")
print("  API Inferencia:  http://localhost:8000/api/v1/infer")
print("  API Cámaras:     http://localhost:8000/api/v1/cameras")
print("  Consola Web:     http://localhost:4321")


Endpoints de Producción Activos:
  API Inferencia:  http://localhost:8000/api/v1/infer
  API Cámaras:     http://localhost:8000/api/v1/cameras
  Consola Web:     http://localhost:4321
